** 代码和文字尚未修正——请忽略！！！ **

## 错误的测量值

如果你能相信你的传感器永远不会发生故障，也不会给出虚假的测量值，那你很幸运，请跳到下一节。现实是传感器并不完美。杂散电压会影响信号，鸟会飞到距离传感器前面，计算机视觉传感器会把阴影误认为行人。等等。

有整本书专门讨论这个主题。我发现那些讨论雷达多目标跟踪的书特别有用。我只谈几点，它们只需要很少的理论和代码，就能大幅改善你的滤波器。

我们讨论过*似然*函数。回顾贝叶斯法则：

$$\text{posterior} = \frac{\text{likelihood}\times\text{prior}}{\text{evidence}}$$

其中似然定义为

$$\mathcal L = p(\mathbf z \mid \mathbf x)$$

这就是在给定先验 $\mathbf x$ 的条件下测量值的似然（概率）。这启发了一个简单的*门限*（gating）函数。这里的*门限*指的是根据某些准则接受或拒绝测量值的函数。我们假设测量噪声是高斯的。回顾高斯分布一章，99.7% 的值落在均值的 3 个标准差之内。如果测量值 $\mathbf z > 3 \sigma$，我们可以认为它极不可能而将其丢弃。很简单。

在实践中你大概不应该这样做。我们把传感器*建模*为高斯的，但实际上它们多半不是。尾部可能较厚，也可能是略有不同的分布，比如学生 t 分布，等等。例如，NASA 在猎户座（Orion）任务中使用 $5\sigma$ 到 $6\sigma$ 来适应传感器的真实行为。你的预算多半比 NASA 少，但我猜你面临的风险也相应小得多。没有办法用解析方法确定正确的值。你需要用自己的数据做实验，看看什么值对你的应用是合理的。

In [ ]:
from math import sqrt

def gated_fusion(pos_data, vel_data, wheel_std, ps_std, gate=3.):
    kf = KalmanFilter(dim_x=2, dim_z=1)
    kf.F = array([[1., 1.], [0., 1.]])
    kf.H = array([[1., 0.], [1., 0.]])
    kf.x = array([[0.], [1.]])
    kf.P *= 100

    xs, ts = [],  []
    
    # copy data for plotting
    zs_wheel = np.array(vel_data)
    zs_ps = np.array(pos_data)
                     
    last_t = 0
    while len(pos_data) > 0 and len(vel_data) > 0:
        if pos_data[0][0] < vel_data[0][0]:
            t, z = pos_data.pop(0)
            dt = t - last_t
            last_t = t
            p_index = 0
            
            kf.H = np.array([[1., 0.]])
            kf.R[0,0] = ps_std**2
            si
        else:
            t, z = vel_data.pop(0)
            dt = t - last_t
            last_t = t
            p_index = 1
            
            kf.H = np.array([[0., 1.]])
            kf.R[0,0] = wheel_std**2

        kf.F[0,1] = dt
        kf.Q = Q_discrete_white_noise(2, dt=dt, var=.02)
        kf.predict()
        std = sqrt(kf.P[p_index, p_index])
        y = abs(kf.residual_of(z)) 
        if  y <= std * gate:
            kf.update(np.array([z]))
        else:
            print('skip', p_index, kf.x.T, kf.P.diagonal(), "%.3f" % z, y)


        xs.append(kf.x.T[0])
        ts.append(t)

    plot_fusion(xs, ts, zs_ps, zs_wheel)
    
random.seed(1123)
pos_data, vel_data = gen_sensor_data(25, 1.5, .3)
gated_fusion(pos_data, vel_data, 1.5, 3.0, gate=4.);
